# Dreamcash vs Tradexyz Tracking Analysis with `polaris_data`

This notebook uses `polaris-data>=0.8.4` to:

- discover the shared `dreamcash` and `tradexyz` markets in Polaris
- fetch a bounded day of 1-minute OHLCV data for a small shared ticker basket
- compare activity, return alignment, and price tracking quality across venues
- zoom into one focus ticker with raw trades and hourly notional plots


## Setup
This repository already pins `polaris-data`, `pandas`, and `matplotlib` in `pyproject.toml`.

Run `make install` once from the repo root, then start JupyterLab with `make notebook`. This notebook keeps the comparison bounded to a single UTC day and a small shared ticker basket so it remains easy to rerun and extend.


In [ ]:
from polaris_data import PolarisClient
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-darkgrid")
pd.options.display.float_format = "{:.4f}".format


In [ ]:
source_a = "dreamcash"
source_b = "tradexyz"

# Use an explicit bounded range with the workspace-pinned polaris-data version.
start = "2026-06-29T00:00:00Z"
end = "2026-06-30T00:00:00Z"

selected_tickers = ["GOLD", "NVDA", "AMZN", "TSLA", "MSFT"]
focus_ticker = "NVDA"


In [ ]:
def normalize_market_id(market_id: str) -> str:
    market_id = market_id.upper()
    for prefix in ("CASH:", "XYZ:"):
        if market_id.startswith(prefix):
            return market_id[len(prefix):]
    return market_id


def bars_to_frame(bars: list[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(bars)
    if frame.empty:
        return pd.DataFrame(columns=["open", "high", "low", "close", "volume", "trades", "interval"])

    frame["timestamp"] = pd.to_datetime(frame["timestamp"], unit="us", utc=True)
    return frame.set_index("timestamp").sort_index()


def trades_to_frame(trades: list[dict]) -> pd.DataFrame:
    frame = pd.json_normalize(trades)
    if frame.empty:
        return pd.DataFrame(columns=["timestamp", "price", "quantity", "side"])

    frame["timestamp"] = pd.to_datetime(frame["timestamp"], unit="us", utc=True)
    frame = frame.rename(
        columns={
            "data.price": "price",
            "data.quantity": "quantity",
            "data.side": "side",
        }
    )
    return frame[["timestamp", "price", "quantity", "side"]].sort_values("timestamp")


def regular_trading_hours_mask(index: pd.DatetimeIndex) -> pd.Series:
    local = index.tz_convert("America/New_York")
    is_weekday = local.dayofweek < 5
    after_open = (local.hour > 9) | ((local.hour == 9) & (local.minute >= 30))
    before_close = local.hour < 16
    return is_weekday & after_open & before_close


def catalog_markets_to_frame(catalog: dict) -> pd.DataFrame:
    frame = pd.DataFrame(catalog["markets"])
    if frame.empty:
        return pd.DataFrame(columns=["source", "market", "start", "end", "categories", "access", "source_type"])
    return frame


## Discover shared markets
Build a normalized ticker map from the Polaris catalog first so each venue comparison uses the correct market ids for the same underlying symbol.


In [ ]:
with PolarisClient() as client:
    catalog = client.catalog()

catalog_markets = catalog_markets_to_frame(catalog)
catalog_markets = catalog_markets[catalog_markets["source"].isin([source_a, source_b])].copy()

shared_market_map = {}
for _, market in catalog_markets.iterrows():
    source_name = market["source"]
    ticker = normalize_market_id(market["market"])
    shared_market_map.setdefault(ticker, {})[source_name] = market["market"]

shared_market_map = {
    ticker: mapping
    for ticker, mapping in shared_market_map.items()
    if source_a in mapping and source_b in mapping
}

available_shared_tickers = sorted(shared_market_map)
missing_tickers = sorted(set(selected_tickers) - set(shared_market_map))
if missing_tickers:
    raise KeyError(f"Selected tickers missing shared markets: {missing_tickers}")

selected_market_map = {ticker: shared_market_map[ticker] for ticker in selected_tickers}

mapping_df = pd.DataFrame(
    [
        {
            "ticker": ticker,
            f"{source_a}_market": mapping[source_a],
            f"{source_b}_market": mapping[source_b],
        }
        for ticker, mapping in selected_market_map.items()
    ]
)

print(f"Shared markets discovered: {len(available_shared_tickers)}")
display(mapping_df)


## Pull OHLCV and score tracking
Fetch 1-minute bars for each selected ticker on both venues, then summarize activity, overlap, and spread behavior across the pair.


In [ ]:
bars_by_ticker = {}

with PolarisClient() as client:
    for ticker, market_map in selected_market_map.items():
        bars_by_ticker[ticker] = {}
        for source_name, market_id in market_map.items():
            bars = client.ohlcv(
                source=source_name,
                market=market_id,
                interval="1m",
                from_=start,
                to=end,
            )
            bars_by_ticker[ticker][source_name] = bars_to_frame(bars)

activity_rows = []
tracking_rows = []

for ticker, pair in bars_by_ticker.items():
    for source_name, frame in pair.items():
        activity_rows.append(
            {
                "ticker": ticker,
                "source": source_name,
                "active_minutes": int(len(frame)),
                "trade_count": int(frame["trades"].sum()) if not frame.empty else 0,
                "volume": float(frame["volume"].sum()) if not frame.empty else 0.0,
                "first_close": float(frame["close"].iloc[0]) if not frame.empty else None,
                "last_close": float(frame["close"].iloc[-1]) if not frame.empty else None,
                "return_pct": float((frame["close"].iloc[-1] / frame["close"].iloc[0] - 1.0) * 100) if len(frame) > 1 else None,
            }
        )

    merged = pair[source_a][["close", "volume", "trades"]].rename(
        columns={
            "close": f"{source_a}_close",
            "volume": f"{source_a}_volume",
            "trades": f"{source_a}_trades",
        }
    ).join(
        pair[source_b][["close", "volume", "trades"]].rename(
            columns={
                "close": f"{source_b}_close",
                "volume": f"{source_b}_volume",
                "trades": f"{source_b}_trades",
            }
        ),
        how="inner",
    )
    merged["spread_bps"] = (merged[f"{source_a}_close"] / merged[f"{source_b}_close"] - 1.0) * 10000

    trade_total = merged[f"{source_a}_trades"].sum() + merged[f"{source_b}_trades"].sum()
    tracking_rows.append(
        {
            "ticker": ticker,
            "overlap_minutes": int(len(merged)),
            "close_corr": float(merged[f"{source_a}_close"].corr(merged[f"{source_b}_close"])) if len(merged) > 1 else None,
            "mean_abs_spread_bps": float(merged["spread_bps"].abs().mean()) if len(merged) else None,
            "max_abs_spread_bps": float(merged["spread_bps"].abs().max()) if len(merged) else None,
            f"{source_a}_share_of_trades": float(merged[f"{source_a}_trades"].sum() / trade_total) if trade_total else None,
        }
    )

activity_summary = pd.DataFrame(activity_rows).sort_values(["ticker", "source"]).reset_index(drop=True)
tracking_metrics = pd.DataFrame(tracking_rows).set_index("ticker").sort_values("mean_abs_spread_bps")

display(activity_summary)
display(tracking_metrics)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

activity_summary.pivot(index="ticker", columns="source", values="trade_count").plot(
    kind="bar",
    ax=axes[0],
    title="Trade count by venue",
)
axes[0].set_ylabel("trades")
axes[0].tick_params(axis="x", rotation=0)

activity_summary.pivot(index="ticker", columns="source", values="active_minutes").plot(
    kind="bar",
    ax=axes[1],
    title="Active 1m bars by venue",
)
axes[1].set_ylabel("minutes")
axes[1].tick_params(axis="x", rotation=0)

tracking_metrics["mean_abs_spread_bps"].sort_values().plot(
    kind="barh",
    ax=axes[2],
    title="Mean absolute close spread",
    color="C2",
)
axes[2].set_xlabel("bps")

plt.tight_layout()


## Zoom into one ticker
After the broad venue scan, plot a single focus ticker to inspect where the minute closes diverge and how large the spread becomes.


In [ ]:
focus_pair = bars_by_ticker[focus_ticker]
focus_merged = focus_pair[source_a][["close", "volume", "trades"]].rename(
    columns={
        "close": f"{source_a}_close",
        "volume": f"{source_a}_volume",
        "trades": f"{source_a}_trades",
    }
).join(
    focus_pair[source_b][["close", "volume", "trades"]].rename(
        columns={
            "close": f"{source_b}_close",
            "volume": f"{source_b}_volume",
            "trades": f"{source_b}_trades",
        }
    ),
    how="inner",
)
focus_merged["spread_bps"] = (focus_merged[f"{source_a}_close"] / focus_merged[f"{source_b}_close"] - 1.0) * 10000

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
focus_merged[[f"{source_a}_close", f"{source_b}_close"]].plot(
    ax=axes[0],
    title=f"{focus_ticker}: minute close comparison",
)
axes[0].set_ylabel("price")

focus_merged["spread_bps"].plot(
    ax=axes[1],
    color="C3",
    title=f"{focus_ticker}: {source_a} vs {source_b} close spread",
)
axes[1].axhline(0.0, color="black", linewidth=1.0)
axes[1].set_ylabel("bps")
axes[1].set_xlabel("timestamp")

plt.tight_layout()

focus_merged.sort_values("spread_bps", key=lambda s: s.abs(), ascending=False).head(10)


## Drill into raw trades
Finish by fetching the raw trade stream for the focus ticker so the example covers both aggregated bars and event-level data from `polaris-data`.


In [ ]:
with PolarisClient() as client:
    focus_trades = {}
    for source_name, market_id in selected_market_map[focus_ticker].items():
        trades = client.trades(
            source=source_name,
            market=market_id,
            from_=start,
            to=end,
        )
        focus_trades[source_name] = trades_to_frame(trades)

trade_summary_rows = []
hourly_notional = {}

for source_name, frame in focus_trades.items():
    trade_count = len(frame)
    notional = frame["price"] * frame["quantity"] if trade_count else pd.Series(dtype=float)
    indexed = frame.set_index("timestamp") if trade_count else frame.copy()

    if trade_count:
        indexed["notional"] = indexed["price"] * indexed["quantity"]
        hourly_notional[source_name] = indexed["notional"].resample("1h").sum()
        rth_mask = regular_trading_hours_mask(indexed.index)
        rth_trade_share = float(indexed.loc[rth_mask].shape[0] / trade_count)
        rth_notional_share = float(indexed.loc[rth_mask, "notional"].sum() / indexed["notional"].sum())
    else:
        hourly_notional[source_name] = pd.Series(dtype=float)
        rth_trade_share = None
        rth_notional_share = None

    trade_summary_rows.append(
        {
            "source": source_name,
            "trade_count": int(trade_count),
            "buy_volume": float(frame.loc[frame["side"] == "buy", "quantity"].sum()) if trade_count else 0.0,
            "sell_volume": float(frame.loc[frame["side"] == "sell", "quantity"].sum()) if trade_count else 0.0,
            "notional_usd": float(notional.sum()) if trade_count else 0.0,
            "rth_trade_share": rth_trade_share,
            "rth_notional_share": rth_notional_share,
        }
    )

focus_trade_summary = pd.DataFrame(trade_summary_rows).set_index("source")
display(focus_trade_summary)


In [ ]:
hourly_notional_df = pd.DataFrame(hourly_notional).fillna(0.0)
ax = hourly_notional_df.plot(
    figsize=(14, 4),
    title=f"{focus_ticker}: hourly traded notional by venue",
)
ax.set_ylabel("notional")
ax.set_xlabel("timestamp")
plt.tight_layout()

for source_name, frame in focus_trades.items():
    print(source_name)
    display(frame.head())


## Notes

- `dreamcash` is much sparser than `tradexyz`, so overlap and spread metrics are driven by the minutes where both venues print prices.
- `tradexyz` behaves like the denser reference market in this notebook, while `dreamcash` is useful for measuring tracking quality and activity concentration.
- To turn this into a wider market scan, expand `selected_tickers` or rank the full shared universe by `mean_abs_spread_bps` and `trade_count`.
